In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

In [8]:
df = pd.read_csv(r'C:\Users\priya\Desktop\PyCh_Pro\Churn_Analysis_and_Modelling\data\raw\telco_churn.csv')

In [10]:
df.sample(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4463,4603-JANFB,Male,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.85,69.85,Yes
4241,2150-UWTFY,Female,0,Yes,Yes,22,Yes,No,DSL,Yes,...,Yes,No,No,No,Month-to-month,No,Mailed check,61.15,1422.05,Yes
2786,6584-VQMYT,Male,0,No,Yes,27,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Mailed check,56.20,1567.55,No
4735,0723-FDLAY,Male,0,No,No,44,Yes,Yes,DSL,No,...,Yes,Yes,Yes,Yes,One year,Yes,Bank transfer (automatic),85.25,3704.15,No
2041,3546-GHEAE,Male,0,No,No,7,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.45,165.35,No


## Started Preparing Data

In [14]:
from sklearn.preprocessing import FunctionTransformer

def preprocessing_raw_data(X):
    df = X.copy()
    internet_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
    ]
    phone_cols = ["MultipleLines"]
    binary_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
        "MultipleLines", "Partner", "Dependents",
        "PhoneService", "PaperlessBilling"
    ]

    
    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
            
    for col in internet_cols:
        if col in df.columns:
            df[col] = df[col].replace("no internet service", "no")
            
    for col in phone_cols:
        if col in df.columns:
            df[col] = df[col].replace("no phone service", "no")
            
    df["Stability"] = df["Partner"].astype(str) + "_" + df["Dependents"].astype(str)
    
    return df

prep_transformer = FunctionTransformer(preprocessing_raw_data)


In [15]:
def binaryEncoder(X):
    df = X.copy()
    binary_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
        "MultipleLines", "Partner", "Dependents",
        "PhoneService", "PaperlessBilling"
    ]
    
    mapping = {"no": 0, "yes": 1}
    print("Encoding Binary Columns to 0 and 1...")
    for col in binary_cols:
        X_train[col] = X_train[col].map(mapping).astype(int)
        X_test[col]  = X_test[col].map(mapping).astype(int)

encoder_transformer = FunctionTransformer(binaryEncoder)

def label_encoder(Y):
    """
    Only for y_train and y_test
    expects target variable only
    """
    df = Y.copy
    df = df.map({"Yes": 1, "No": 0})

    return df

In [17]:
def precision_recall_at_k(y_test, y_prob, return_precision = True, k = 0.2):
    """
    Compute Recall@K and Precision@K for ranking-based evaluation.
    
    Parameters:
    - y_test: true labels (0/1)
    - y_prob: predicted probabilities
    - k: fraction of top samples to consider
    
    Returns:
    - recall_at_k, precision_at_k
    """
    df_eval = pd.DataFrame({
        "prob" : y_prob,
        "actual" : y_test.values
    })
    df_eval = df_eval.sort_values(by = "prob", ascending = False)

    top_k = max(1, int(k * len(df_eval)))
    top_churners = df_eval.head(top_k)
    
    total_churn = df_eval["actual"].sum()
    if total_churn == 0:
        return (0, 0) if return_precision else 0
    recall = top_churners["actual"].sum() / total_churn
    # out of all the churners, how many churns are catched in the top 20%
    if return_precision:
        precision = top_churners["actual"].mean() if len(top_churners) > 0 else 0
        # in the top 20% range, how many are actual churns.
        return recall, precision

    return recall

## Building pipeline for Classification